# Projection and Residuals — Contextual Embeddings

Projects Hebrew and Arabic contextual embedding spaces onto the English space
using least-squares regression, then computes the residuals.

## What this does

For each foreign language (Hebrew, Arabic):
1. Find W such that `English @ W ≈ Foreign` (least-squares)
2. Compute projected: `English @ W`
3. Compute residual: `Foreign - projected`

The **residual** captures what Hebrew/Arabic encodes that English cannot explain.
This is the cross-lingual semantic signal tested in the encoding model.

## Input files
- `en_sliding_window_embeddings.csv` — (1735, 768) English contextual embeddings
- `he_sliding_window_embeddings.csv` — (1735, 768) Hebrew contextual embeddings
- `ar_sliding_window_embeddings.csv` — (1735, 768) Arabic contextual embeddings

## Output files
- `hebrew_residuals_contextual.npy`  — (1735, 768)
- `arabic_residuals_contextual.npy`  — (1735, 768)
- `hebrew_projected_contextual.npy`  — (1735, 768)
- `arabic_projected_contextual.npy`  — (1735, 768)

## 1. Load Embeddings

In [1]:
import numpy as np
import pandas as pd

# ── EMBEDDING MODE ────────────────────────────────────────────────────────────
# Set this before running. Must match what you set in notebook 06.
#
#   'sliding_window'  — 1735 words, 32-word context window (notebook 04)
#   'contextual'      — 1692 words, full sentence context  (notebooks 01–03)
#
EMBEDDING_MODE = 'contextual'

DATA_DIR = '../data/processed/'

if EMBEDDING_MODE == 'sliding_window':
    print('Loading sliding-window embeddings (notebook 04)...')
    E = pd.read_csv(DATA_DIR + 'en_sliding_window_embeddings.csv').values.astype(float)
    H = pd.read_csv(DATA_DIR + 'he_sliding_window_embeddings.csv').values.astype(float)
    A = pd.read_csv(DATA_DIR + 'ar_sliding_window_embeddings.csv').values.astype(float)

elif EMBEDDING_MODE == 'contextual':
    print('Loading contextual-aligned embeddings (notebooks 01–03)...')
    en_idx   = pd.read_csv(DATA_DIR + 'en_contextual_matched_indices.csv')
    orig_idx = en_idx['original_word_idx'].values          # 1692 word indices
    E = pd.read_csv(DATA_DIR + 'en_contextual_aligned_embeddings.csv').values.astype(float)
    H = pd.read_csv(DATA_DIR + 'he_contextual_aligned_embeddings.csv').values[orig_idx].astype(float)
    A = pd.read_csv(DATA_DIR + 'ar_contextual_aligned_embeddings.csv').values[orig_idx].astype(float)

else:
    raise ValueError(f'Unknown EMBEDDING_MODE: {EMBEDDING_MODE!r}. Choose sliding_window or contextual.')

print(f'\nMode    : {EMBEDDING_MODE}')
print(f'English : {E.shape}')
print(f'Hebrew  : {H.shape}')
print(f'Arabic  : {A.shape}')
assert E.shape == H.shape == A.shape, 'Shape mismatch between languages!'
print('\nAll shapes verified ✓')

Loading contextual-aligned embeddings (notebooks 01–03)...

Mode    : contextual
English : (1692, 768)
Hebrew  : (1692, 768)
Arabic  : (1692, 768)

All shapes verified ✓


## 2. Projection and Residual

In [2]:
def project_and_residual(X_foreign, X_english):
    """
    Find W such that X_english @ W ≈ X_foreign (least squares).
    
    Projected = X_english @ W  (the part of foreign explainable by English)
    Residual  = X_foreign - Projected  (what English cannot explain)
    
    The residual is the cross-lingual semantic signal:
    information in Hebrew/Arabic that is orthogonal to English.
    """
    W, _, _, _ = np.linalg.lstsq(X_english, X_foreign, rcond=None)
    X_projected = X_english @ W
    X_residual  = X_foreign - X_projected
    return X_projected, X_residual


print('Computing Hebrew projection and residual...')
H_projected, H_residual = project_and_residual(H, E)
print(f'  Hebrew projected : {H_projected.shape}')
print(f'  Hebrew residual  : {H_residual.shape}')

print('\nComputing Arabic projection and residual...')
A_projected, A_residual = project_and_residual(A, E)
print(f'  Arabic projected : {A_projected.shape}')
print(f'  Arabic residual  : {A_residual.shape}')

Computing Hebrew projection and residual...
  Hebrew projected : (1692, 768)
  Hebrew residual  : (1692, 768)

Computing Arabic projection and residual...
  Arabic projected : (1692, 768)
  Arabic residual  : (1692, 768)


## 3. Sanity Checks

In [3]:
# The residual should be orthogonal to English (by construction)
# Check: correlation between English and residuals should be near 0

def mean_correlation(X, Y):
    """Mean absolute correlation between columns of X and Y."""
    X_norm = (X - X.mean(0)) / (X.std(0) + 1e-8)
    Y_norm = (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
    corr = (X_norm * Y_norm).mean()
    return corr

print('Sanity checks:')
print(f'  Mean corr(English, Hebrew_residual)  : {mean_correlation(E, H_residual):.4f}  (should be ~0)')
print(f'  Mean corr(English, Arabic_residual)  : {mean_correlation(E, A_residual):.4f}  (should be ~0)')
print(f'  Mean corr(English, Hebrew)           : {mean_correlation(E, H):.4f}  (baseline)')
print(f'  Mean corr(English, Arabic)           : {mean_correlation(E, A):.4f}  (baseline)')

# Check variance explained by projection
he_var_explained = 1 - np.var(H_residual) / np.var(H)
ar_var_explained = 1 - np.var(A_residual) / np.var(A)
print(f'\n  Variance of Hebrew explained by English : {he_var_explained*100:.1f}%')
print(f'  Variance of Arabic explained by English : {ar_var_explained*100:.1f}%')
print(f'  (Residual contains the remaining {(1-he_var_explained)*100:.1f}% / {(1-ar_var_explained)*100:.1f}%)')

Sanity checks:
  Mean corr(English, Hebrew_residual)  : 0.0000  (should be ~0)
  Mean corr(English, Arabic_residual)  : -0.0000  (should be ~0)
  Mean corr(English, Hebrew)           : 0.1710  (baseline)
  Mean corr(English, Arabic)           : 0.2322  (baseline)

  Variance of Hebrew explained by English : 99.2%
  Variance of Arabic explained by English : 99.7%
  (Residual contains the remaining 0.8% / 0.3%)


## 4. Save

In [4]:
np.save(DATA_DIR + f'hebrew_residuals_{EMBEDDING_MODE}.npy',  H_residual)
np.save(DATA_DIR + f'arabic_residuals_{EMBEDDING_MODE}.npy',  A_residual)
np.save(DATA_DIR + f'hebrew_projected_{EMBEDDING_MODE}.npy',  H_projected)
np.save(DATA_DIR + f'arabic_projected_{EMBEDDING_MODE}.npy',  A_projected)

print(f'Saved (mode = {EMBEDDING_MODE}):')
print(f'  hebrew_residuals_{EMBEDDING_MODE}.npy   {H_residual.shape}')
print(f'  arabic_residuals_{EMBEDDING_MODE}.npy   {A_residual.shape}')
print(f'  hebrew_projected_{EMBEDDING_MODE}.npy   {H_projected.shape}')
print(f'  arabic_projected_{EMBEDDING_MODE}.npy   {A_projected.shape}')
print(f'\nNext step: set EMBEDDING_MODE = {EMBEDDING_MODE!r} in notebook 06 and run the encoding loop.')

Saved (mode = contextual):
  hebrew_residuals_contextual.npy   (1692, 768)
  arabic_residuals_contextual.npy   (1692, 768)
  hebrew_projected_contextual.npy   (1692, 768)
  arabic_projected_contextual.npy   (1692, 768)

Next step: set EMBEDDING_MODE = 'contextual' in notebook 06 and run the encoding loop.


In [5]:
import numpy as np
import pandas as pd

def parse_embedding(x):
    if isinstance(x, str):
        x = x.replace("[", "").replace("]", "")
        return np.array(x.split(), dtype=float)
    return np.array(x, dtype=float)

df = pd.read_csv('../data/Amirim_Project_Submission/podcast_trilingual_embeddings.csv')

E = np.vstack(df["en_embedding"].apply(parse_embedding).to_numpy()).astype(np.float32)
H = np.vstack(df["he_embedding"].apply(parse_embedding).to_numpy()).astype(np.float32)
A = np.vstack(df["ar_embedding"].apply(parse_embedding).to_numpy()).astype(np.float32)

def project_and_residual(X_foreign, X_english):
    W, _, _, _ = np.linalg.lstsq(X_english, X_foreign, rcond=None)
    X_projected = X_english @ W
    X_residual  = X_foreign - X_projected
    return X_projected, X_residual

H_proj, H_res = project_and_residual(H, E)
A_proj, A_res = project_and_residual(A, E)

# Report variance
he_var = 1 - np.var(H_res) / np.var(H)
ar_var = 1 - np.var(A_res) / np.var(A)
print(f"FastText Hebrew  — variance explained by English: {he_var*100:.1f}%  residual: {(1-he_var)*100:.1f}%")
print(f"FastText Arabic  — variance explained by English: {ar_var*100:.1f}%  residual: {(1-ar_var)*100:.1f}%")

# Save
np.save('../data/processed/fasttext_hebrew_residuals.npy',  H_res.astype(np.float32))
np.save('../data/processed/fasttext_arabic_residuals.npy',  A_res.astype(np.float32))
np.save('../data/processed/fasttext_hebrew_projected.npy',  H_proj.astype(np.float32))
np.save('../data/processed/fasttext_arabic_projected.npy',  A_proj.astype(np.float32))
print("Saved FastText residuals.")

FastText Hebrew  — variance explained by English: 51.0%  residual: 49.0%
FastText Arabic  — variance explained by English: 53.4%  residual: 46.6%
Saved FastText residuals.
